# Notebook 04e — RL Training: N=1, 2, 3 (REINFORCE + Baseline)

Trains SimplexPolicy with REINFORCE+value-baseline at N=1 (SPY), N=2 (SPY+IEF),
N=3 (SPY+IEF+GLD). 10 seeds × 200 episodes each. Evaluates the mean-across-seeds
policy on the test window (2023–2026).

**Training window:** 2018-01-01 → 2022-12-31 (5-year window, ≈1260 days).  
Regime diversity is preserved (2020 COVID crash, 2021 recovery, 2022 rate-hike drawdown).

**Truncated rollouts (CPU):** `MAX_STEPS_PER_EPISODE=60` — each training episode uses
60-step windows drawn from the start of the training window. This cuts per-seed time
from ~2 min to ~3s and makes all 30 seed runs complete in ~1.5 min.  
For full rollouts on GPU (Colab), set `MAX_STEPS_PER_EPISODE=None`.

**Expected outcomes (sanity, not paper-grade):**
- N=1: net Sharpe ≈ always-SPY (~1.4). Policy has no freedom; documents the trivial case.
- N=2: trained Sharpe > random rollout. May or may not beat always-SPY.
- N=3: turnover > 0; Sharpe competitive with best single asset or ≥ 1/N baseline.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import warnings
warnings.filterwarnings('ignore')

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path

from aiam.data.split import TRAIN_END, TEST_START
from aiam.rl.env import PortfolioEnv
from aiam.rl.policy import SimplexPolicy
from aiam.rl.agent import RLAgent
from aiam.rl.trainer import TrainConfig, run_episode

SEEDS = list(range(10))
LOOKBACK = 20
EPISODES = 200
# 60-step truncated episodes for fast CPU runs (~1.5 min total).
# Set to None for full rollouts on GPU (Colab) → ~19 min.
MAX_STEPS_PER_EPISODE = 60
TRAIN_START = pd.Timestamp('2018-01-01')  # 5-year window: COVID + rate-hike diversity
TRADING_DAYS = 252
RESULTS_DIR = Path('../results/rl')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Seeds: {SEEDS}')
print(f'Episodes per seed: {EPISODES}  |  Steps per episode: {MAX_STEPS_PER_EPISODE}')
print(f'Training window: {TRAIN_START.date()} → {TRAIN_END.date()}')
print(f'Results: {RESULTS_DIR}')

In [ ]:
returns_full = pd.read_parquet('../data/cache/returns_29_2003_2026.parquet').dropna()
returns_train = returns_full.loc[
    (returns_full.index >= TRAIN_START) & (returns_full.index <= TRAIN_END)
]
returns_test = returns_full.loc[returns_full.index >= TEST_START]

ASSET_SETS = {
    1: ['SPY.US'],
    2: ['SPY.US', 'IEF.US'],
    3: ['SPY.US', 'IEF.US', 'GLD.US'],
}

print(f'Train: {returns_train.index[0].date()} → {returns_train.index[-1].date()} ({len(returns_train)} days)')
print(f'Test : {returns_test.index[0].date()} → {returns_test.index[-1].date()} ({len(returns_test)} days)')

In [ ]:
def compute_sharpe(returns_series: np.ndarray) -> float:
    r = np.array(returns_series)
    if r.std() < 1e-10:
        return 0.0
    return float(r.mean() / r.std() * np.sqrt(TRADING_DAYS))


def compute_max_drawdown(returns_series: np.ndarray) -> float:
    cumret = np.cumprod(1 + np.array(returns_series))
    peak = np.maximum.accumulate(cumret)
    dd = (cumret - peak) / peak
    return float(dd.min())


def eval_policy_on_env(
    policy: SimplexPolicy,
    env: PortfolioEnv,
    stochastic: bool = False,
    seed: int = 0,
) -> dict:
    torch.manual_seed(seed)
    np.random.seed(seed)

    if stochastic:
        rewards, _, _, turnovers, weights_list = run_episode(policy, None, env, temperature=1.0)
    else:
        state = env.reset()
        done = False
        rewards, turnovers, weights_list = [], [], []
        while not done:
            w = policy.act(state)
            state, r, done, info = env.step(w)
            rewards.append(r)
            turnovers.append(info.get('turnover', 0.0))
            weights_list.append(w)

    r_arr = np.array(rewards)
    w_arr = np.array(weights_list)
    return {
        'sharpe': compute_sharpe(r_arr),
        'vol': float(r_arr.std() * np.sqrt(TRADING_DAYS)),
        'max_drawdown': compute_max_drawdown(r_arr),
        'turnover_annual': float(np.mean(turnovers) * TRADING_DAYS),
        'mean_weights': w_arr.mean(axis=0),
        'rewards': r_arr,
        'weights': w_arr,
    }


def eval_single_asset(rets: pd.DataFrame, asset: str) -> dict:
    r = rets[asset].values
    return {
        'sharpe': compute_sharpe(r),
        'vol': float(r.std() * np.sqrt(TRADING_DAYS)),
        'max_drawdown': compute_max_drawdown(r),
        'turnover_annual': 0.0,
    }


def eval_equal_weight(rets: pd.DataFrame, assets: list) -> dict:
    r = rets[assets].mean(axis=1).values
    return {
        'sharpe': compute_sharpe(r),
        'vol': float(r.std() * np.sqrt(TRADING_DAYS)),
        'max_drawdown': compute_max_drawdown(r),
        'turnover_annual': 0.0,
    }


print('Helpers defined.')

In [ ]:
# ============================================================
# Main training loop: N = 1, 2, 3
# ============================================================
all_results = {}
t_wall_start = time.time()

for n_assets, assets in ASSET_SETS.items():
    print(f'\n{"="*60}')
    print(f'N={n_assets}: {assets}')
    print('='*60)

    out_dir = RESULTS_DIR / f'n{n_assets}'
    out_dir.mkdir(parents=True, exist_ok=True)

    train_env = PortfolioEnv(returns_train[assets], lookback=LOOKBACK)
    test_env  = PortfolioEnv(returns_test[assets],  lookback=LOOKBACK)

    per_seed_metrics = []
    all_policies = []
    all_histories = []

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        policy = SimplexPolicy(
            n_features=train_env.n_features,
            hidden_dim=32,
            use_weights=True,
        )
        agent = RLAgent(policy, lookback=LOOKBACK, seed=seed)
        config = TrainConfig(
            episodes=EPISODES,
            gamma=0.95,
            lr=1e-3,
            entropy_coef=0.01,
            grad_clip=1.0,
            seed=seed,
            max_steps_per_episode=MAX_STEPS_PER_EPISODE,
        )

        t0 = time.time()
        history = agent.fit(train_env, config, use_value_baseline=True)
        elapsed = time.time() - t0

        # Save policy
        agent.save(str(out_dir / f'agent_seed_{seed}.pt'))
        all_policies.append(agent.policy)
        all_histories.append(history)

        m = eval_policy_on_env(agent.policy, test_env, stochastic=False, seed=seed)
        per_seed_metrics.append(m)

        last20 = np.mean(history.episode_rewards[-20:])
        print(f'  Seed {seed:2d}: ep_rwd(last20)={last20:+.4f}  '
              f'test_sharpe={m["sharpe"]:>6.3f}  ({elapsed:.1f}s)')

    # Ensemble: mean-weight policy across seeds
    ensemble_policy = SimplexPolicy(
        n_features=train_env.n_features, hidden_dim=32, use_weights=True
    )
    state_dicts = [p.state_dict() for p in all_policies]
    avg_state = {
        k: torch.stack([sd[k].float() for sd in state_dicts]).mean(0)
        for k in state_dicts[0]
    }
    ensemble_policy.load_state_dict(avg_state)
    ensemble_policy.eval()
    m_ensemble = eval_policy_on_env(ensemble_policy, test_env, stochastic=False, seed=0)

    # Random (untrained) baseline
    torch.manual_seed(999)
    random_policy = SimplexPolicy(n_features=train_env.n_features, hidden_dim=32, use_weights=True)
    np.random.seed(999)
    m_random = eval_policy_on_env(random_policy, test_env, stochastic=True, seed=999)

    baselines = {f'BH({a})': eval_single_asset(returns_test, a) for a in assets}
    baselines['EW(1/N)'] = eval_equal_weight(returns_test, assets)

    all_results[n_assets] = {
        'assets': assets,
        'per_seed': per_seed_metrics,
        'ensemble': m_ensemble,
        'random': m_random,
        'baselines': baselines,
        'histories': all_histories,
    }

print(f'\nTotal wall time: {(time.time() - t_wall_start)/60:.1f} min')
print('Training complete.')

In [ ]:
# ============================================================
# Training curves (10-episode rolling mean)
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (n_assets, res) in zip(axes, all_results.items()):
    for i, history in enumerate(res['histories']):
        ep_r = pd.Series(history.episode_rewards).rolling(10, min_periods=1).mean()
        ax.plot(ep_r.values, lw=0.6, alpha=0.5, label=f'seed {i}' if i < 3 else None)
    ax.set_title(f'N={n_assets}: Training Curves (10-ep MA)')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Total Episode Reward')
    if n_assets == 1:
        ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('../results/rl/training_curves.png', dpi=120)
plt.show()
print('Training curves saved.')

In [ ]:
# ============================================================
# Comparison tables
# ============================================================

def fmt_row(label, m):
    return (f'{label:<30s} {m.get("sharpe", float("nan")):>7.3f}'
            f' {m.get("vol", float("nan")):>7.3f}'
            f' {m.get("max_drawdown", float("nan")):>8.3f}'
            f' {m.get("turnover_annual", float("nan")):>10.1f}')

for n_assets, res in all_results.items():
    print(f'\n{"="*70}')
    print(f'N={n_assets} — {res["assets"]}  |  Test period 2023–2026')
    print('='*70)
    print(f'{"Strategy":<30s} {"Sharpe":>7s} {"Vol":>7s} {"MaxDD":>8s} {"TO/yr":>10s}')
    print('-' * 70)

    sharpes = [m['sharpe'] for m in res['per_seed']]
    print(f'{"Trained (10 seeds mean)":<30s} {np.mean(sharpes):>7.3f} (±{np.std(sharpes):.3f})')
    print(fmt_row('Trained (ensemble avg)', res['ensemble']))
    print(fmt_row('Untrained (random rollout)', res['random']))
    print('-' * 70)
    for name, m in res['baselines'].items():
        print(fmt_row(name, m))

    ens = res['ensemble']['sharpe']
    rnd = res['random']['sharpe']
    ew  = res['baselines']['EW(1/N)']['sharpe']
    print(f'\n  Ensemble vs random: {"✓" if ens > rnd else "✗"}  |  '
          f'Ensemble vs EW: {"✓" if ens > ew else "✗"}')
    if n_assets == 1:
        print('  NOTE: N=1 has zero degrees of freedom — Sharpe ≈ BH(SPY) is expected.')
    if n_assets == 3 and res['ensemble']['turnover_annual'] < 5:
        print('  NOTE: Annual turnover < 5 — policy near-static. Review training curves.')

In [ ]:
# ============================================================
# Weight allocation charts (ensemble policy, N=2 and N=3)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, n_assets in zip(axes, [2, 3]):
    res = all_results[n_assets]
    w_arr = res['ensemble']['weights']
    for i, a in enumerate(res['assets']):
        ax.plot(w_arr[:, i], lw=0.8, label=a.replace('.US', ''))
    ax.axhline(1/n_assets, color='k', lw=0.5, ls='--', label='1/N')
    ax.set_title(f'N={n_assets} Ensemble Policy — Test Window Weights')
    ax.set_xlabel('Trading Day')
    ax.set_ylabel('Weight')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../results/rl/ensemble_weights.png', dpi=120)
plt.show()

In [ ]:
# ============================================================
# Cumulative return comparison (N=2 and N=3)
# ============================================================
for n_assets in [2, 3]:
    res = all_results[n_assets]
    assets = res['assets']
    r_ens = res['ensemble']['rewards']

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(np.cumprod(1 + r_ens), lw=1.5, label='Trained (ensemble)', color='steelblue')
    ax.plot(np.cumprod(1 + res['random']['rewards']), lw=1.0, ls='--',
            label='Untrained (random)', color='gray')

    colors = ['tomato', 'seagreen', 'darkorange']
    for j, a in enumerate(assets):
        r_bh = returns_test[a].values[:len(r_ens)]
        ax.plot(np.cumprod(1 + r_bh), lw=1.0, ls=':',
                label=f'BH({a.replace(".US", "")})', color=colors[j % len(colors)])

    ax.set_title(f'N={n_assets} Cumulative Return — Test Window 2023–2026')
    ax.set_xlabel('Trading Day')
    ax.set_ylabel('Cumulative Return')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(f'../results/rl/cumret_n{n_assets}.png', dpi=120)
    plt.show()

print('All figures saved to results/rl/')

In [ ]:
# ============================================================
# Session 4b verdict
# ============================================================
print('SESSION 4b VERDICT')
print('='*60)
for n_assets, res in all_results.items():
    sharpes = [m['sharpe'] for m in res['per_seed']]
    ens = res['ensemble']['sharpe']
    print(f'N={n_assets}:')
    print(f'  10-seed Sharpe: {np.mean(sharpes):.3f} ± {np.std(sharpes):.3f}')
    print(f'  Ensemble Sharpe: {ens:.3f}')
    print(f'  Max DD: {res["ensemble"]["max_drawdown"]:.3f}')
    print(f'  Turnover/yr: {res["ensemble"]["turnover_annual"]:.1f}')
    print()

print('Sharpe bar to clear in full 29-asset run: 2.579 (ML ensemble, Session 2)')
print('Next: Session 4c — 29-asset evaluation + walk-forward + PPO/SAC upgrade if needed')